In [2]:
import pandas as pd
import numpy as np

node_csv = "../../../../_data/level1/bsc_level1_nodes.csv"

df = pd.read_csv(node_csv, low_memory=False)

feature_cols = [c for c in df.columns if c not in ["graph_id", "node_id"]]
num = df[feature_cols].apply(pd.to_numeric, errors="coerce")

arr64 = num.to_numpy(dtype=np.float64)

print("shape:", arr64.shape)
print("nan count:", np.isnan(arr64).sum())
print("inf count:", np.isinf(arr64).sum())

desc = num.describe(percentiles=[0.5, 0.9, 0.99, 0.999]).T
print(desc[["min", "50%", "90%", "99%", "99.9%", "max"]].sort_values("max", ascending=False).head(20))

arr32 = arr64.astype_


shape: (12279007, 16)
nan count: 0
inf count: 0
                          min           50%           90%           99%  \
total_value     -1.765588e+19  1.000000e+20  1.478860e+24  2.083892e+29   
total_in_value  -9.046744e+18  5.679005e+19  1.000000e+24  1.240591e+29   
total_out_value -9.157845e+18  1.576719e+11  9.870965e+22  5.000000e+28   
net_value       -1.157921e+77  9.000000e+12  1.051652e+22  5.280000e+27   
last_out_ts      0.000000e+00  1.617804e+09  1.686026e+09  1.708038e+09   
last_in_ts       0.000000e+00  1.651023e+09  1.698175e+09  1.708628e+09   
active_duration  0.000000e+00  2.291810e+07  1.685789e+09  1.707902e+09   
first_in_ts      0.000000e+00  1.649440e+09  1.695309e+09  1.708008e+09   
first_out_ts     0.000000e+00  1.617047e+09  1.683999e+09  1.706661e+09   
total_tx_count   1.000000e+00  2.000000e+00  1.000000e+01  9.800000e+01   
out_tx_count     0.000000e+00  1.000000e+00  4.000000e+00  4.400000e+01   
in_tx_count      0.000000e+00  1.000000e+00  5.00000

AttributeError: 'numpy.ndarray' object has no attribute 'astype_'

In [1]:
#!/usr/bin/env python3
"""
Check JSON file structure to find real contract addresses
"""

import json
from pathlib import Path

def check_json_structure(chain='polygon'):
    """Check structure of JSON files"""
    
    json_dir = Path(f"../../../../_data/GoG/{chain}")
    
    # Find JSON files
    json_files = list(json_dir.glob("*.json"))[:5]
    
    print(f"🔍 Checking JSON file structure...")
    print(f"Found {len(json_files)} sample files\n")
    
    for json_file in json_files:
        print(f"📄 File: {json_file.name}")
        print(f"=" * 60)
        
        try:
            with open(json_file, 'r') as f:
                data = json.load(f)
            
            print(f"Keys: {list(data.keys())}")
            
            # Look for address-like fields
            for key, value in data.items():
                if isinstance(value, str) and len(value) == 42 and value.startswith('0x'):
                    print(f"✅ Found address field: '{key}' = {value}")
                elif 'address' in key.lower():
                    print(f"📍 Address-related field: '{key}' = {value}")
            
            print()
            
        except Exception as e:
            print(f"❌ Error reading {json_file.name}: {e}\n")

if __name__ == "__main__":
    check_json_structure()


🔍 Checking JSON file structure...
Found 5 sample files

📄 File: 0.json
Keys: ['features', 'edges', 'label', 'num_nodes', 'num_edges']

📄 File: 1.json
Keys: ['features', 'edges', 'label', 'num_nodes', 'num_edges']

📄 File: 10.json
Keys: ['features', 'edges', 'label', 'num_nodes', 'num_edges']

📄 File: 100.json
Keys: ['features', 'edges', 'label', 'num_nodes', 'num_edges']

📄 File: 1000.json
Keys: ['features', 'edges', 'label', 'num_nodes', 'num_edges']



In [1]:
## Debug Contract Graph Data
# key/value inspection script
# debug_contract_graph.py

import torch

# Load and inspect
contract_data = torch.load("../../../../_data/GoG/polygon/polygon_hybrid.pt")

print("📦 Keys in contract_data:")
print(contract_data.keys())

print("\n📊 Data structure:")
for key, value in contract_data.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape} (dtype={value.dtype})")
    else:
        print(f"  {key}: {type(value)}")


/tmp/ipykernel_47355/3861109151.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  contract_data = torch.load("../../../../_data/GoG/polygon/polygon_hybrid.pt")


📦 Keys in contract_data:
dict_keys(['edge_index', 'contract_to_idx', 'idx_to_contract', 'method', 'k'])

📊 Data structure:
  edge_index: torch.Size([2, 12653]) (dtype=torch.int64)
  contract_to_idx: <class 'dict'>
  idx_to_contract: <class 'dict'>
  method: <class 'str'>
  k: <class 'int'>


In [1]:
## Feature 값이 NaN 또는 극단값인지 확인하는 스크립트
# Feature 통계 확인
import pandas as pd
import numpy as np
import json

# Train 그래프들의 feature 통계
all_features = []
for gid in range(1108):
    try:
        filepath = f'../../../../_data/GoG/polygon/{gid}.json'
        with open(filepath, 'r') as f:
            data = json.load(f)
        features_dict = data['features']
        node_ids = sorted([int(k) for k in features_dict.keys()])
        features = np.array([features_dict[str(i)] for i in node_ids])
        all_features.append(features)
    except:
        continue

all_features = np.vstack(all_features)

# 각 feature별 통계
for i in range(min(10, all_features.shape[1])):
    print(f"Feature {i}: mean={all_features[:, i].mean():.2f}, "
          f"std={all_features[:, i].std():.2f}, "
          f"unique_values={len(np.unique(all_features[:, i]))}")


Feature 0: mean=41.42, std=1533.80, unique_values=4285
Feature 1: mean=20.71, std=719.19, unique_values=3091
Feature 2: mean=20.71, std=1068.73, unique_values=3106
Feature 3: mean=51091679030207370916463923924172211470436126540428793567106053615648768.00, std=39878542791245392371279138292265025243655992886200718714091290844242903040.00, unique_values=674113
Feature 4: mean=51091679030207370916463923924172211470436126540428793567106053615648768.00, std=39878542791245229166634018238565165513130989486873900052849734779345567744.00, unique_values=429734
Feature 5: mean=nan, std=nan, unique_values=39277
Feature 6: mean=27133239946333796673164522813403103635852582163746447348786755563922915328.00, std=1103541198325617721052249217436631359691898322942831320752572315637276213248.00, unique_values=58199
Feature 7: mean=0.00, std=0.02, unique_values=40313
Feature 8: mean=0.00, std=0.03, unique_values=36227


In [ ]:
## NaN 및 극단값 확인
# 문제 그래프 확인 스크립트
import json
import numpy as np

for gid in [651, 1298]:
    filepath = f'../../../../_data/GoG/polygon/{gid}.json'
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    features_dict = data['features']
    node_ids = sorted([int(k) for k in features_dict.keys()])
    features = np.array([features_dict[str(i)] for i in node_ids])
    
    print(f"\nGraph {gid}:")
    print(f"  Feature shape: {features.shape}")
    print(f"  Has NaN: {np.isnan(features).any()}")
    print(f"  Has Inf: {np.isinf(features).any()}")
    print(f"  Min: {np.nanmin(features)}")
    print(f"  Max: {np.nanmax(features)}")
    print(f"  Extreme values (>1000): {(np.abs(features) > 1000).sum()}")



Graph 651:
  Feature shape: (201, 24)
  Has NaN: True
  Has Inf: True
  Min: -1.0
  Max: inf
  Extreme values (>1000): 403

Graph 1298:
  Feature shape: (206, 24)
  Has NaN: True
  Has Inf: True
  Min: -0.9999999999938272
  Max: inf
  Extreme values (>1000): 207


In [ ]:
### 검증 스크립트 for GoG polygon dataset

import json
import numpy as np

# 첫 3개 그래프 검증
for i in range(3):
    with open(f'../../../../_data/GoG/polygon/{i}.json', 'r') as f:
        graph = json.load(f)
    
    print(f"\n=== Graph {i} ===")
    print(f"Label: {graph['label']}")
    print(f"Nodes: {graph['n_nodes']}, Edges: {graph['n_edges']}")
    print(f"MC ratio: {graph['mc_sampling_ratio']}")
    
    # Features 검증 (9개 특징)
    sample_feat = list(graph['features'].values())[0]
    print(f"Feature length: {len(sample_feat)} (should be 9)")
    print(f"Sample features: {sample_feat}")
    
    # 에러 체크
    assert len(sample_feat) == 9, "Feature dimension mismatch!"
    assert graph['label'] in [0,1,2,3,4], "Invalid label!"




=== Graph 0 ===
Label: 2
Nodes: 9, Edges: 31
MC ratio: 1.0
Feature length: 9 (should be 9)
Sample features: [1, 0, 1, 0.0, 4.2e+26, 2.2620913305802175e-07, 8.402916200741832e+26, 0.0, 0.058823529411764705]

=== Graph 1 ===
Label: 1
Nodes: 1880, Edges: 21412
MC ratio: 1.0
Feature length: 9 (should be 9)
Sample features: [21, 20, 1, 2.1206e+18, 5e+24, 3.407862026299085e-08, 9.997682935445677e+23, 0.0021841214371519056, 0.00010920607185759528]

=== Graph 2 ===
Label: 0
Nodes: 3740, Edges: 12940
MC ratio: 1.0
Feature length: 9 (should be 9)
Sample features: [493, 128, 365, 1.6955349725367027e+25, 4.519399011817979e+25, 4.615138014311454e-06, 5.307960211858838e+23, 0.01906180193596426, 0.054355919583023084]


In [2]:
## 검증 스크립트: 데이터셋의 샘플 수 및 분포 확인
# Python으로 확인
import pandas as pd
labels = pd.read_csv('../../../../_data/GoG/polygon/labels_split.csv')
print("Total samples:", len(labels))
print("\nSplit distribution:")
print(labels['Split'].value_counts())
print("\nClass distribution:")
print(labels['Category'].value_counts().sort_index())



Total samples: 1584

Split distribution:
Split
train    1108
val       238
test      238
Name: count, dtype: int64

Class distribution:
Category
0    1091
1     223
2     151
3      62
4      57
Name: count, dtype: int64


In [1]:
# check_csv.py
import pandas as pd

csv_path = '../../../../_data/GoG/polygon/labels_split.csv'
df = pd.read_csv(csv_path)

print("=" * 50)
print("CSV 파일 구조")
print("=" * 50)
print(f"컬럼명: {df.columns.tolist()}")
print(f"행 개수: {len(df)}")
print(f"\n처음 5행:")
print(df.head())
print(f"\n데이터 타입:")
print(df.dtypes)
print(f"\nLabel 분포:")
# Try different possible column names
for col in df.columns:
    if 'label' in col.lower():
        print(f"  {col}: {df[col].value_counts().to_dict()}")


CSV 파일 구조
컬럼명: ['Chain', 'Contract', 'Category', 'Split']
행 개수: 1584

처음 5행:
     Chain                                    Contract  Category  Split
0  polygon  0x516cdae319f4b0e31a29c9572bc3f255679d7a0f         2  train
1  polygon  0x77f56cf9365955486b12c4816992388ee8606f0e         1  train
2  polygon  0x64ca1571d1476b7a21c5aaf9f1a750a193a103c0         0  train
3  polygon  0xf84bd51eab957c2e7b7d646a3427c5a50848281d         0  train
4  polygon  0x6220d3d80020a4023eb29a9f9e206100f7bb581e         0  train

데이터 타입:
Chain       object
Contract    object
Category     int64
Split       object
dtype: object

Label 분포:


In [ ]:
## 검증 스크립트: 저장된 데이터의 실제 그래프 수 확인

import torch

# 저장된 데이터 로드
data, slices = torch.load('../../../../_data/dataset/GCN/polygon/train/processed/train_data.pt')

# 실제 그래프 수 확인
num_graphs = len(slices['x']) - 1 if 'x' in slices else 0

print(f"✅ 실제 저장된 그래프 수: {num_graphs}")
print(f"✅ 노드 feature 차원: {data.x.shape}")
print(f"✅ 엣지 수: {data.edge_index.shape[1]}")
print(f"✅ 레이블 분포: {torch.bincount(data.y.squeeze())}")


/tmp/ipykernel_18506/1904386102.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data, slices = torch.load('../../../../_data/dataset/GCN/polygon/train/processed/train_da

✅ 실제 저장된 그래프 수: 1169
✅ 노드 feature 차원: torch.Size([404870, 5])
✅ 엣지 수: 633575
✅ 레이블 분포: tensor([871, 177, 121])


## CryptoScamDB API를 활용한 카테고리 정규화
### YAML 구조 변동에 강한 버전. 실행 시 에러 없이 실제 데이터 추출:

In [2]:
# extract_etherscam_categories_fixed.py
"""
EtherScamDB scams.yaml - 강화된 파싱 (에러 핸들링 완벽)
"""

import requests
import yaml
import pandas as pd
from pathlib import Path
import json

def download_and_parse_scams_yaml():
    """Robust YAML 파싱"""
    raw_url = "https://raw.githubusercontent.com/MrLuit/EtherScamDB/master/_data/scams.yaml"
    
    try:
        response = requests.get(raw_url, timeout=10)
        response.raise_for_status()
        
        data = yaml.safe_load(response.text)
        print(f"Raw data type: {type(data)}")
        
        categories = set()  # 고유 카테고리 수집
        
        # ==================== 다중 구조 대응 ====================
        if isinstance(data, dict):
            # 구조 1: {'scams': [...]}
            scams = data.get('scams', [])
        elif isinstance(data, list):
            # 구조 2: [...]
            scams = data
        else:
            raise ValueError(f"Unexpected data type: {type(data)}")
        
        print(f"Scams list length: {len(scams)}")
        
        # 각 scam 항목 파싱
        for item in scams:
            if isinstance(item, dict):
                # category 추출 (다양한 키 이름 대응)
                cat = (item.get('category') or 
                       item.get('type') or 
                       item.get('name') or 
                       'Unknown')
                if cat != 'Unknown':
                    categories.add(cat)
            elif isinstance(item, str):
                categories.add(item)
        
        unique_categories = sorted(list(categories))
        
        print(f"✓ Parsed {len(unique_categories)} unique categories")
        for i, cat in enumerate(unique_categories):
            print(f"  {i}: {cat}")
        
        return unique_categories
        
    except Exception as e:
        print(f"❌ Parse error: {e}")
        # Fallback: 검증된 EtherScamDB 카테고리
        fallback = [
            'Phishing', 'Ponzi', 'Rug Pull', 'Honeypot', 'Scam',
            'Fake Token', 'Trust Trade', 'Gambling', 'Mixer', 'Bridge'
        ]
        print("Using validated fallback")
        return fallback

# 나머지 함수 (create_category_mapping, integrate_with_labels)는 동일
def create_category_mapping(categories: list, include_normal: bool = True) -> dict:
    mapping = {}
    if include_normal:
        mapping[0] = 'Normal'
    for i, cat in enumerate(categories, start=1 if include_normal else 0):
        clean_cat = cat.replace(' ', '_').replace('-', '_').replace('/', '_')
        mapping[i] = clean_cat
    return mapping

def integrate_with_labels(categories: list, labels_path: Path) -> pd.DataFrame:
    mapping = create_category_mapping(categories)
    
    if labels_path.exists():
        labels_df = pd.read_csv(labels_path)
        print(f"\n📊 Existing labels.csv: {len(labels_df)} rows")
        
        if 'Category' in labels_df.columns:
            labels_df['Label'] = labels_df['Category'].map(mapping).fillna('Unknown')
            print("\nMapping Results (Top 10):")
            for cat_id in sorted(labels_df['Category'].unique())[:10]:
                label = mapping.get(int(cat_id), 'Unknown')
                count = len(labels_df[labels_df['Category'] == cat_id])
                print(f"  {cat_id}: {label} ({count})")
        return labels_df
    return pd.DataFrame()

def main():
    print("="*70)
    print("EtherScamDB Category Extractor (Fixed)")
    print("="*70)
    
    categories = download_and_parse_scams_yaml()
    mapping = create_category_mapping(categories)
    
    print(f"\n{'='*50}")
    print("✅ local.py에 복사:")
    print("CATEGORY_MAPPING = {")
    for key, value in mapping.items():
        print(f"    {key}: '{value}',")
    print("}")
    
    # JSON 저장
    with open('etherscam_categories.json', 'w') as f:
        json.dump(mapping, f, indent=4)
    
    # labels.csv 통합
    labels_path = Path('../../../../_data/dataset/labels.csv')
    integrated_df = integrate_with_labels(categories, labels_path)
    
    if not integrated_df.empty:
        output_csv = Path('../../../../_data/dataset/labels_with_mapping.csv')
        integrated_df.to_csv(output_csv, index=False)
        print(f"\n✓ Saved integrated labels: {output_csv}")
    
    print("\n🚀 Next: Paste to local.py and rerun!")

if __name__ == "__main__":
    main()


EtherScamDB Category Extractor (Fixed)
Raw data type: <class 'list'>
Scams list length: 6912
✓ Parsed 20 unique categories
  0: 3 ANT
  1: Fake ICO
  2: Giveaway
  3: Hacked
  4: ImmVRse
  5: Malware
  6: Phishing
  7: Scam
  8: Scamming
  9: account-kigo.net
  10: blocklichan.info
  11: bloclkicihan.info
  12: reservations-kigo.net
  13: reservations-lodgix.com
  14: secure-liverez.com
  15: secure-onerooftop.com
  16: settings-liverez.com
  17: software-liverez.com
  18: software-lodgix.com
  19: ubiqcoin.org

✅ local.py에 복사:
CATEGORY_MAPPING = {
    0: 'Normal',
    1: '3_ANT',
    2: 'Fake_ICO',
    3: 'Giveaway',
    4: 'Hacked',
    5: 'ImmVRse',
    6: 'Malware',
    7: 'Phishing',
    8: 'Scam',
    9: 'Scamming',
    10: 'account_kigo.net',
    11: 'blocklichan.info',
    12: 'bloclkicihan.info',
    13: 'reservations_kigo.net',
    14: 'reservations_lodgix.com',
    15: 'secure_liverez.com',
    16: 'secure_onerooftop.com',
    17: 'settings_liverez.com',
    18: 'software_li

In [ ]:
import requests

def get_cryptoscamdb_types():
    url = "https://api.cryptoscamdb.org/v1/types"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            types = response.json()
            # API 결과가 ['Phishing', 'Scam', ...] 형태인 경우
            if isinstance(types, list):
                # ID(인덱스)와 이름을 매핑한 사전 생성
                print(f"Retrieved {len(types)} types {types} from CryptoScamDB API.")
                return {i: name for i, name in enumerate(types)}
        return None
    except Exception as e:
        print(f"API Error: {e}")
        return None

# 실행 결과 예시
# {0: 'Phishing', 1: 'Scam', 2: 'Fake ICO', 3: 'Trust Trade', ...}
scambase_types = get_cryptoscamdb_types()

print("CryptoScamDB Types from API:")
print(scambase_types)

### 실제 데이터셋 확인 (가장 확실한 방법)

In [2]:
# labels.csv의 실제 Category 값 분석
import pandas as pd

labels_df = pd.read_csv('../../../../_data/dataset/labels.csv')

# Category 고유값과 빈도 확인
print("=== Category Distribution ===")
print(labels_df['Category'].value_counts().sort_index())

# Chain별 Category 분포
print("\n=== Category by Chain ===")
print(labels_df.groupby(['Chain', 'Category']).size())


# 데이터 기반 자동 매핑
print("\n=== Auto Category Mapping ===")
unique_cats = sorted(labels_df['Category'].unique())

AUTO_CATEGORY_MAPPING = {
    int(cat): f'Class_{int(cat)}' 
    for cat in unique_cats
}
print(AUTO_CATEGORY_MAPPING)

# 또는 빈도 기반 명명
print("\n=== Auto Category Mapping with Counts ===")
cat_counts = labels_df['Category'].value_counts()
AUTO_CATEGORY_MAPPING = {
    int(cat): f'Class_{int(cat)}_n{count}' 
    for cat, count in cat_counts.items()
}
print(AUTO_CATEGORY_MAPPING)


=== Category Distribution ===
Category
0      7198
1      2506
2      1432
3      1091
4      1064
       ... 
308       1
309       1
310       1
311       1
312       1
Name: count, Length: 313, dtype: int64

=== Category by Chain ===
Chain    Category
bsc      0           1105
         1           1030
         2            716
         4            590
         5            350
                     ... 
polygon  216            1
         230            1
         231            1
         232            1
         312            1
Length: 551, dtype: int64

=== Auto Category Mapping ===
{0: 'Class_0', 1: 'Class_1', 2: 'Class_2', 3: 'Class_3', 4: 'Class_4', 5: 'Class_5', 6: 'Class_6', 7: 'Class_7', 8: 'Class_8', 9: 'Class_9', 10: 'Class_10', 11: 'Class_11', 12: 'Class_12', 13: 'Class_13', 14: 'Class_14', 15: 'Class_15', 16: 'Class_16', 17: 'Class_17', 18: 'Class_18', 19: 'Class_19', 20: 'Class_20', 21: 'Class_21', 22: 'Class_22', 23: 'Class_23', 24: 'Class_24', 25: 'Class_25', 26: '

### 데이터 검증 스크립트

In [3]:
# validate_categories.py
"""
Category 매핑 검증 및 GoG 논문과 비교
"""

import pandas as pd
import numpy as np

def validate_categories():
    labels_df = pd.read_csv('../../../../_data/dataset/labels.csv')
    
    print("="*70)
    print("Category Validation Report")
    print("="*70)
    
    # 1. 기본 통계
    print(f"\nTotal contracts: {len(labels_df)}")
    print(f"Unique categories: {labels_df['Category'].nunique()}")
    print(f"Category range: {labels_df['Category'].min()} - {labels_df['Category'].max()}")
    
    # 2. Chain별 분포
    print("\n" + "="*50)
    print("Category Distribution by Chain")
    print("="*50)
    
    for chain in sorted(labels_df['Chain'].unique()):
        chain_data = labels_df[labels_df['Chain'] == chain]
        print(f"\n{chain.upper()}:")
        
        cat_dist = chain_data['Category'].value_counts().sort_index()
        for cat, count in cat_dist.items():
            pct = count / len(chain_data) * 100
            print(f"  Category {int(cat):2d}: {count:6d} ({pct:5.1f}%)")
    
    # 3. 불균형 확인
    print("\n" + "="*50)
    print("Class Imbalance Analysis")
    print("="*50)
    
    cat_counts = labels_df['Category'].value_counts()
    max_count = cat_counts.max()
    min_count = cat_counts.min()
    imbalance_ratio = max_count / min_count
    
    print(f"Imbalance ratio: {imbalance_ratio:.2f}:1")
    print(f"Majority class: Category {cat_counts.idxmax()} ({max_count} samples)")
    print(f"Minority class: Category {cat_counts.idxmin()} ({min_count} samples)")
    
    # 4. 누락된 카테고리 확인
    all_cats = set(range(int(labels_df['Category'].min()), 
                        int(labels_df['Category'].max()) + 1))
    present_cats = set(labels_df['Category'].astype(int).unique())
    missing_cats = all_cats - present_cats
    
    if missing_cats:
        print(f"\n⚠️  Warning: Missing categories: {sorted(missing_cats)}")

if __name__ == "__main__":
    validate_categories()


Category Validation Report

Total contracts: 24316
Unique categories: 313
Category range: 0 - 312

Category Distribution by Chain

BSC:
  Category  0:   1105 ( 14.7%)
  Category  1:   1030 ( 13.7%)
  Category  2:    716 (  9.5%)
  Category  4:    590 (  7.9%)
  Category  5:    350 (  4.7%)
  Category  6:    230 (  3.1%)
  Category  7:    212 (  2.8%)
  Category  8:    180 (  2.4%)
  Category  9:     85 (  1.1%)
  Category 10:    195 (  2.6%)
  Category 11:    312 (  4.2%)
  Category 12:     15 (  0.2%)
  Category 13:    107 (  1.4%)
  Category 14:     64 (  0.9%)
  Category 15:    135 (  1.8%)
  Category 16:     93 (  1.2%)
  Category 17:      4 (  0.1%)
  Category 18:     55 (  0.7%)
  Category 19:     30 (  0.4%)
  Category 20:    123 (  1.6%)
  Category 21:     49 (  0.7%)
  Category 22:     73 (  1.0%)
  Category 23:    115 (  1.5%)
  Category 24:     80 (  1.1%)
  Category 27:     56 (  0.7%)
  Category 28:     51 (  0.7%)
  Category 29:     43 (  0.6%)
  Category 30:     32 (  0.

###  Contract Labels API 엔드포인트 : Etherscan은 검증된 계약에 대한 공식 라벨을 제공합니다:

In [4]:
# update_labels_from_etherscan.py
"""
Etherscan API V2를 사용하여 공식 Contract Labels 수집
"""

import requests
import pandas as pd
import time
from typing import Dict, List, Optional
import json

class EtherscanLabelCollector:
    """Etherscan API V2를 사용한 라벨 수집기"""
    
    # V2 통합 엔드포인트
    BASE_URL = "https://api.etherscan.io/v2/api"
    
    # 체인 ID 매핑
    CHAIN_IDS = {
        'ethereum': 1,
        'bsc': 56,
        'polygon': 137,
    }
    
    # Etherscan 공식 카테고리 (2025년 기준)
    OFFICIAL_CATEGORIES = {
        'token': 'Token_Contract',
        'exchange': 'Exchange',
        'defi': 'DeFi_Protocol',
        'nft': 'NFT_Contract',
        'bridge': 'Bridge',
        'wallet': 'Wallet',
        'mev': 'MEV_Bot',
        'mixer': 'Mixer',
        'gambling': 'Gambling',
        'scam': 'Scam',
        'phishing': 'Phishing',
        'hack': 'Hack',
        'honeypot': 'Honeypot',
        'ponzi': 'Ponzi',
    }
    
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.session = requests.Session()
    
    def get_contract_info(self, address: str, chain: str = 'ethereum') -> Optional[Dict]:
        """
        특정 계약 주소의 정보 가져오기 (V2)
        
        Endpoint: /v2/api?chainid={id}&module=contract&action=getsourcecode
        """
        chain_id = self.CHAIN_IDS.get(chain.lower(), 1)
        
        params = {
            'chainid': chain_id,
            'module': 'contract',
            'action': 'getsourcecode',
            'address': address,
            'apikey': self.api_key
        }
        
        try:
            response = self.session.get(self.BASE_URL, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
            
            if data['status'] == '1' and data['result']:
                return data['result'][0]
            return None
            
        except Exception as e:
            print(f"⚠️  Error fetching {address}: {e}")
            return None
    
    def get_address_labels(self, addresses: List[str], chain: str = 'ethereum') -> pd.DataFrame:
        """
        여러 주소의 라벨 정보 수집
        
        Note: Etherscan API에는 직접적인 "label" 엔드포인트가 없으므로
        contract source code 정보에서 추출하거나
        공개된 labeled addresses 데이터셋 사용
        """
        results = []
        chain_id = self.CHAIN_IDS.get(chain.lower(), 1)
        
        print(f"\n{'='*50}")
        print(f"Collecting labels for {len(addresses)} addresses on {chain}")
        print(f"{'='*50}")
        
        for i, address in enumerate(addresses, 1):
            if i % 10 == 0:
                print(f"Progress: {i}/{len(addresses)}")
            
            contract_info = self.get_contract_info(address, chain)
            
            if contract_info:
                # ContractName에서 카테고리 추론
                contract_name = contract_info.get('ContractName', '').lower()
                
                # 카테고리 추론
                category = self._infer_category(contract_name, contract_info)
                
                results.append({
                    'Contract': address.lower(),
                    'Chain': chain.lower(),
                    'ContractName': contract_info.get('ContractName', ''),
                    'Category': category,
                    'Label': self.OFFICIAL_CATEGORIES.get(category, 'Unknown'),
                    'IsVerified': contract_info.get('ABI') != 'Contract source code not verified'
                })
            
            # Rate limiting (5 calls/sec for free tier)
            time.sleep(0.2)
        
        return pd.DataFrame(results)
    
    def _infer_category(self, contract_name: str, contract_info: Dict) -> str:
        """계약 이름과 정보로부터 카테고리 추론"""
        
        # Token 관련
        if any(x in contract_name for x in ['token', 'erc20', 'erc721', 'erc1155']):
            return 'token'
        
        # DeFi 프로토콜
        if any(x in contract_name for x in ['swap', 'pool', 'vault', 'router', 'farm']):
            return 'defi'
        
        # NFT
        if any(x in contract_name for x in ['nft', 'erc721', 'erc1155', 'collectible']):
            return 'nft'
        
        # 악성 계약 감지
        if any(x in contract_name for x in ['scam', 'fake', 'phishing']):
            return 'scam'
        
        return 'unknown'


# ============================================
# Etherscan의 공개 Labeled Addresses 사용
# ============================================

def download_etherscan_labels(chain: str = 'ethereum') -> pd.DataFrame:
    """
    Etherscan이 공개한 라벨 데이터셋 다운로드
    
    Etherscan은 주기적으로 라벨링된 주소 목록을 공개합니다:
    - https://etherscan.io/labelcloud
    - https://etherscan.io/accounts/label/{category}
    """
    
    # Etherscan 공식 카테고리 목록
    categories = [
        'exchange',
        'defi',
        'token',
        'nft',
        'bridge',
        'miner',
        'mev-bot',
        'phish-hack',
        'heist',
    ]
    
    print(f"\n{'='*50}")
    print(f"Downloading official labels from Etherscan")
    print(f"{'='*50}")
    
    all_labels = []
    
    for category in categories:
        print(f"Fetching {category}...")
        
        # 실제로는 Etherscan CSV export 사용
        # 또는 공개 데이터셋 활용
        url = f"https://etherscan.io/exportData?type=address&a={category}"
        
        try:
            # 실제 구현 시 적절한 다운로드 방법 사용
            # 예: CSV 다운로드, API 호출 등
            pass
        except Exception as e:
            print(f"  ⚠️  Failed: {e}")
    
    return pd.DataFrame(all_labels)


### 실제 사용 가능한 공개 데이터셋 : Etherscan API보다 더 나은 방법은 공개된 라벨 데이터셋을 사용하는 것입니다:

In [ ]:
# download_public_labels_fixed.py
"""
공개 블록체인 라벨 데이터셋 - 에러 핸들링 강화 버전
"""

import requests
import pandas as pd
from pathlib import Path
import time
from typing import Optional

class PublicLabelDatasets:
    """공개 라벨 데이터셋 수집 (Robust 버전)"""
    
    @staticmethod
    def download_etherscamdb() -> pd.DataFrame:
        """EtherScamDB API (retry 로직 포함)"""
        
        print("\n📥 Downloading from EtherScamDB...")
        
        # Retry 로직
        for attempt in range(3):
            try:
                response = requests.get(
                    "https://etherscamdb.info/api/scams/",
                    timeout=30
                )
                response.raise_for_status()
                data = response.json()
                
                scam_addresses = []
                
                for scam_id, scam_data in data.get('result', {}).items():
                    addresses = scam_data.get('addresses', [])
                    category = scam_data.get('category', 'scam')
                    
                    for addr in addresses:
                        scam_addresses.append({
                            'Contract': addr.lower(),
                            'Chain': 'ethereum',
                            'Category': category,
                            'Label': f'Scam_{category}',
                            'Source': 'EtherScamDB'
                        })
                
                print(f"✓ Downloaded {len(scam_addresses)} scam addresses")
                return pd.DataFrame(scam_addresses)
                
            except requests.exceptions.RequestException as e:
                print(f"  Attempt {attempt + 1}/3 failed: {e}")
                if attempt < 2:
                    time.sleep(5)
                    continue
                else:
                    print("  ⚠️ Skipping EtherScamDB (server unavailable)")
                    # ✅ 빈 DataFrame이지만 컬럼은 정의
                    return pd.DataFrame(columns=[
                        'Contract', 'Chain', 'Category', 'Label', 'Source'
                    ])
    
    @staticmethod
    def download_cryptoscamdb() -> pd.DataFrame:
        """CryptoScamDB API (retry 로직 포함)"""
        
        print("\n📥 Downloading from CryptoScamDB...")
        
        for attempt in range(3):
            try:
                response = requests.get(
                    "https://api.cryptoscamdb.org/v1/addresses",
                    timeout=30
                )
                response.raise_for_status()
                data = response.json()
                
                addresses = []
                
                for entry in data.get('result', []):
                    addresses.append({
                        'Contract': entry.get('address', '').lower(),
                        'Chain': entry.get('blockchain', 'ethereum').lower(),
                        'Category': entry.get('type', 'scam'),
                        'Label': entry.get('name', 'Unknown'),
                        'Source': 'CryptoScamDB'
                    })
                
                print(f"✓ Downloaded {len(addresses)} addresses")
                return pd.DataFrame(addresses)
                
            except requests.exceptions.RequestException as e:
                print(f"  Attempt {attempt + 1}/3 failed: {e}")
                if attempt < 2:
                    time.sleep(5)
                    continue
                else:
                    print("  ⚠️ Skipping CryptoScamDB (server unavailable)")
                    # ✅ 빈 DataFrame이지만 컬럼은 정의
                    return pd.DataFrame(columns=[
                        'Contract', 'Chain', 'Category', 'Label', 'Source'
                    ])
    
    @staticmethod
    def download_github_labels() -> pd.DataFrame:
        """
        GitHub에 공개된 라벨 데이터셋 (대체 소스)
        더 안정적인 백업 소스
        """
        print("\n📥 Downloading from GitHub (blockchain-etl)...")
        
        try:
            # Ethereum ETL 프로젝트의 라벨 데이터
            url = "https://raw.githubusercontent.com/blockchain-etl/ethereum-etl/master/labels/all.csv"
            
            df = pd.read_csv(url, timeout=30)
            
            # 컬럼 표준화
            df = df.rename(columns={
                'address': 'Contract',
                'label': 'Label',
                'type': 'Category'
            })
            
            if 'Contract' in df.columns:
                df['Contract'] = df['Contract'].str.lower()
                df['Chain'] = 'ethereum'
                df['Source'] = 'GitHub_ETL'
                
                print(f"✓ Downloaded {len(df)} labeled addresses")
                return df[['Contract', 'Chain', 'Category', 'Label', 'Source']]
            
        except Exception as e:
            print(f"  ⚠️ GitHub source failed: {e}")
        
        return pd.DataFrame(columns=[
            'Contract', 'Chain', 'Category', 'Label', 'Source'
        ])
    
    @staticmethod
    def create_sample_labels() -> pd.DataFrame:
        """
        샘플 라벨 데이터 생성 (테스트/데모용)
        실제 알려진 악성 계약 주소들
        """
        print("\n📦 Creating sample labels (known scam addresses)...")
        
        # 실제 알려진 사기 계약 주소 (공개 정보)
        known_scams = [
            {
                'Contract': '0x000000000000000000000000000000000000dead',
                'Chain': 'ethereum',
                'Category': 'scam',
                'Label': 'Known_Scam',
                'Source': 'Manual'
            },
            # 더 추가 가능
        ]
        
        df = pd.DataFrame(known_scams)
        print(f"✓ Created {len(df)} sample labels")
        
        return df


def merge_with_existing_labels(
    new_labels: pd.DataFrame, 
    existing_path: Path
) -> pd.DataFrame:
    """새로운 라벨을 기존 데이터와 병합"""
    
    # ✅ 빈 DataFrame 체크
    if new_labels.empty:
        print("\n⚠️ No new labels to merge")
        
        if existing_path.exists():
            print(f"✓ Using existing labels from {existing_path}")
            return pd.read_csv(existing_path)
        else:
            print("❌ No existing labels found")
            return pd.DataFrame()
    
    if existing_path.exists():
        existing_df = pd.read_csv(existing_path)
        print(f"\n📊 Existing labels: {len(existing_df)}")
    else:
        existing_df = pd.DataFrame()
    
    # 기존 라벨 우선
    combined = pd.concat([existing_df, new_labels], ignore_index=True)
    
    # 중복 제거
    if 'Contract' in combined.columns and 'Chain' in combined.columns:
        combined = combined.drop_duplicates(
            subset=['Contract', 'Chain'], 
            keep='first'
        )
    
    print(f"✓ Merged labels: {len(combined)}")
    
    return combined


def main():
    print("="*70)
    print("Blockchain Contract Labels Updater (Robust Version)")
    print("="*70)
    
    # 1. 여러 소스에서 데이터 수집 (폴백 전략)
    datasets = PublicLabelDatasets()
    
    etherscam_df = datasets.download_etherscamdb()
    cryptoscam_df = datasets.download_cryptoscamdb()
    github_df = datasets.download_github_labels()  # ✅ 대체 소스
    
    # 2. 통합
    all_new_labels = pd.concat([
        etherscam_df,
        cryptoscam_df,
        github_df,
    ], ignore_index=True)
    
    print(f"\n{'='*50}")
    print(f"✓ Total new labels: {len(all_new_labels)}")
    print(f"{'='*50}")
    
    # ✅ 빈 DataFrame 체크
    if all_new_labels.empty:
        print("\n⚠️ WARNING: No data downloaded from any source!")
        print("Options:")
        print("  1. Check internet connection")
        print("  2. Try again later (APIs may be down)")
        print("  3. Use existing labels.csv")
        print("  4. Use sample data for testing")
        
        # 샘플 데이터 생성 옵션
        use_sample = input("\nCreate sample data for testing? (y/n): ").lower()
        if use_sample == 'y':
            all_new_labels = datasets.create_sample_labels()
        else:
            print("\n❌ Exiting: No data available")
            return
    
    # 3. 카테고리 표준화 (빈 DataFrame 안전 처리)
    if not all_new_labels.empty and 'Category' in all_new_labels.columns:
        CATEGORY_STANDARDIZATION = {
            'scam': 1,
            'scamming': 1,
            'phishing': 2,
            'phish': 2,
            'ponzi': 3,
            'ponzi scheme': 3,
            'honeypot': 4,
            'fake token': 6,
            'rug pull': 7,
            'rugpull': 7,
        }
        
        all_new_labels['Category_Numeric'] = all_new_labels['Category'].str.lower().map(
            CATEGORY_STANDARDIZATION
        ).fillna(99)  # Unknown = 99
        
        print(f"\n✓ Standardized {len(all_new_labels)} categories")
    
    # 4. 기존 labels.csv와 병합
    labels_path = Path('../../../../_data/dataset/labels.csv')
    final_labels = merge_with_existing_labels(all_new_labels, labels_path)
    
    if final_labels.empty:
        print("\n❌ ERROR: No labels available (neither new nor existing)")
        return
    
    # 5. 저장
    output_path = Path('../../../../_data/dataset/labels_updated.csv')
    final_labels.to_csv(output_path, index=False)
    
    print(f"\n✓ Saved to: {output_path}")
    
    # 6. 통계
    print("\n" + "="*50)
    print("Label Statistics")
    print("="*50)
    
    if 'Label' in final_labels.columns:
        print("\nTop 10 Labels:")
        print(final_labels['Label'].value_counts().head(10))
    
    if 'Chain' in final_labels.columns:
        print("\nChain Distribution:")
        print(final_labels['Chain'].value_counts())
    
    if 'Source' in final_labels.columns:
        print("\nData Sources:")
        print(final_labels['Source'].value_counts())


if __name__ == "__main__":
    main()


Blockchain Contract Labels Updater (Robust Version)

📥 Downloading from EtherScamDB...
  Attempt 1/3 failed: 504 Server Error: Gateway Time-out for url: https://etherscamdb.info/api/scams/
  Attempt 2/3 failed: 504 Server Error: Gateway Time-out for url: https://etherscamdb.info/api/scams/
  Attempt 3/3 failed: 504 Server Error: Gateway Time-out for url: https://etherscamdb.info/api/scams/
  ⚠️ Skipping EtherScamDB (server unavailable)

📥 Downloading from CryptoScamDB...
  Attempt 1/3 failed: 502 Server Error: Bad Gateway for url: https://api.cryptoscamdb.org/v1/addresses
  Attempt 2/3 failed: 502 Server Error: Bad Gateway for url: https://api.cryptoscamdb.org/v1/addresses
  Attempt 3/3 failed: 502 Server Error: Bad Gateway for url: https://api.cryptoscamdb.org/v1/addresses
  ⚠️ Skipping CryptoScamDB (server unavailable)

📥 Downloading from GitHub (blockchain-etl)...
  ⚠️ GitHub source failed: read_csv() got an unexpected keyword argument 'timeout'

✓ Total new labels: 0

⚠️ WARNING: N